In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

# Load pretrained ResNet18
resnet = models.resnet18()

# Replace final layer for CIFAR-100 (100 classes)
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 100)


In [5]:
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# Data transforms
transform = transforms.Compose([
    transforms.Resize(224),   # ResNet expects 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# CIFAR-100 dataset
train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [6]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)

# Training loop
for epoch in range(5):  # adjust epochs
    resnet.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 3.7321
Epoch 2, Loss: 2.1985
Epoch 3, Loss: 2.2568
Epoch 4, Loss: 1.6180
Epoch 5, Loss: 1.3486


In [7]:
resnet.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)


Test Accuracy: 52.88


In [8]:
from torchvision.datasets import ImageFolder

local_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

local_dataset = ImageFolder(root="data/pytorchTraining", transform=local_transform)
train_loader = DataLoader(local_dataset, batch_size=32, shuffle=True)

# Update final layer for local dataset classes
num_classes = len(local_dataset.classes)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

# Retrain with same loop


In [9]:
print(local_dataset.classes)

['0', '1', '2']


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)

# Training loop
for epoch in range(5):  # adjust epochs
    resnet.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7569
Epoch 2, Loss: 0.2751
Epoch 3, Loss: 0.0713
Epoch 4, Loss: 0.0200
Epoch 5, Loss: 0.0147


In [14]:
from PIL import Image


def preprocess_image(path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Option A: using PIL
    img = Image.open(path).convert("RGB")
    tensor = transform(img).unsqueeze(0)  # add batch dimension
    tensor = tensor.to(device)
    return tensor

In [15]:
def predict_image(path, model, class_names=None):
    input_tensor = preprocess_image(path)
    with torch.no_grad():
        outputs = model(input_tensor)
        _, pred = torch.max(outputs, 1)
    label = pred.item()
    if class_names:
        return class_names[label]
    return label

In [29]:
class_names = ["0","1","2"]
results = predict_image("data/pytorchTraining/1/110.jpeg", resnet, class_names)
print(results)


2
